# LeafScan AI - 5-class training (Corn Rust / Gray Leaf Spot / Healthy / Northern Leaf Blight / Not Corn Leaf)

**Data:** only `leafscan-5class.zip` from Google Drive (class folders + `groups.csv`).
**Split:** 70 : 15 : 15 per class, seed 42, by *group* (`groups.csv`) so near-identical photos never land in different subsets.
**Model:** MobileNetV2 transfer learning (head first, then fine-tune), class weights, EarlyStopping + ModelCheckpoint + ReduceLROnPlateau.
**Augmentation:** flip / rotation / zoom / brightness / contrast (inside the model, as before) **+ random crop** (training data pipeline only).
**Reports:** accuracy, precision, recall, F1, confusion matrix on the held-out test split.
**Exports** (same files and `labels.json` format as before) go to a NEW Drive folder: `.keras`, `.tflite`, `labels.json`.
**Last cell:** scores your real field photos (never used for training) and counts how many real corn photos are rejected as `not_corn`.

Before running: Runtime > Change runtime type > **T4 GPU**, then edit the paths in the next cell.

In [ ]:
# ===== 1. SETTINGS - edit the 3 paths, then Runtime > Run all =====
ZIP_PATH       = "/content/drive/MyDrive/leafscan-5class.zip"   # the zip you uploaded to Drive
FIELD_TEST_DIR = "/content/drive/MyDrive/field-test"            # real corn photos, TEST ONLY (used by the last cell)
EXPORT_DIR     = "/content/drive/MyDrive/export_5class_v2"      # NEW folder - the old export_5class is NOT touched
MODEL_VERSION  = "mobilenetv2-5class-v2"                        # written into labels.json (max 50 chars)
QUICK_TEST     = False                                          # True = tiny run, only to check the notebook works

import os, sys, io, csv, json, time, random, shutil, zipfile, hashlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import tensorflow as tf
import keras
from PIL import Image, ImageOps

try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content"
except ImportError:
    WORK = os.path.abspath("work")
    os.makedirs(WORK, exist_ok=True)
    print("Not running in Colab: Drive is not mounted.")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

# The ORDER of this list is the order of the model outputs (and of labels.json).
CLASS_NAMES = ["Corn Rust", "Gray Leaf Spot", "Healthy", "Northern Leaf Blight", "Not Corn Leaf"]
NC = CLASS_NAMES.index("Not Corn Leaf")
IMG, BATCH = (224, 224), 32
SPLIT = (0.70, 0.15, 0.15)      # train / validation / test
if QUICK_TEST:
    EXPORT_DIR += "_quicktest"  # never overwrite a real export with a smoke-test run

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - Runtime > Change runtime type > T4 GPU")

## 2. Unzip and list the images

In [ ]:
DATA_DIR = f"{WORK}/leafscan_data"
shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(DATA_DIR)
assert os.path.exists(ZIP_PATH), f"Zip not found: {ZIP_PATH}  (fix ZIP_PATH in cell 1)"

# Manual extraction: accepts both "a/b" and "a\b" entry names (zips made on Windows) and blocks path escapes.
base = os.path.abspath(DATA_DIR)
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        if info.is_dir() or name.endswith("/"):
            continue
        target = os.path.abspath(os.path.join(base, name))
        assert target.startswith(base + os.sep), f"unsafe path in zip: {info.filename}"
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)

# The zip may or may not have one top-level folder: find the folder that holds the 5 class folders.
ROOT = next((dp for dp, dn, _ in os.walk(DATA_DIR) if all(c in dn for c in CLASS_NAMES)), None)
assert ROOT, f"Could not find the 5 class folders {CLASS_NAMES} inside the zip"
extra_dirs = sorted(set(d for d in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, d))) - set(CLASS_NAMES))
if extra_dirs:
    print("Ignoring folders that are not classes:", extra_dirs)

IMG_EXT = (".jpg", ".jpeg", ".png")
files, labels, skipped = [], [], Counter()
for ci, c in enumerate(CLASS_NAMES):
    for dp, _, fns in os.walk(os.path.join(ROOT, c)):
        for fn in sorted(fns):
            if fn.lower().endswith(IMG_EXT):
                files.append(os.path.join(dp, fn)); labels.append(ci)
            else:
                skipped[os.path.splitext(fn)[1].lower() or "(no extension)"] += 1
labels = np.array(labels)
print("Skipped non-image files:", dict(skipped) or "none")
print("class_names =", CLASS_NAMES)
for ci, c in enumerate(CLASS_NAMES):
    print(f"  {ci}  {c:<22}{int((labels == ci).sum()):>6} images")
print(f"  total{int(len(files)):>30}")

## 3. Split 70 : 15 : 15 by group (seed 42)
`groups.csv` puts every image of the same original photo (and near-identical photos) into one group; a group is never split.
Files that are not in `groups.csv` (for example new photos) fall back to the part of the filename before `.rf.`, or to the whole filename.

In [ ]:
GROUPS_CSV = os.path.join(ROOT, "groups.csv")
group_of = {}
if os.path.exists(GROUPS_CSV):
    with open(GROUPS_CSV, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            group_of[(r["class"], r["filename"])] = r["group"]
    print(f"groups.csv: {len(group_of)} rows")
else:
    print("WARNING: groups.csv not found - falling back to the .rf. filename prefix only")

def group_id(cls, path):
    fn = os.path.basename(path)
    return group_of.get((cls, fn)) or f"{cls}::{fn.split('.rf.')[0]}"

missing = sum(1 for p, l in zip(files, labels) if (CLASS_NAMES[l], os.path.basename(p)) not in group_of)
print("images not listed in groups.csv (fallback grouping):", missing)

split = np.full(len(files), -1)          # 0 = train, 1 = validation, 2 = test
rng = random.Random(SEED)
for ci, c in enumerate(CLASS_NAMES):
    idx = np.where(labels == ci)[0]
    groups = defaultdict(list)
    for i in idx:
        groups[group_id(c, files[i])].append(i)
    keys = sorted(groups)                # sort first so the shuffle is reproducible
    rng.shuffle(keys)
    done = 0
    for k in keys:
        frac = done / len(idx)
        s = 0 if frac < SPLIT[0] else 1 if frac < SPLIT[0] + SPLIT[1] else 2
        split[groups[k]] = s
        done += len(groups[k])

# Integrity: a group must live in exactly one subset
seen = defaultdict(set)
for i, p in enumerate(files):
    seen[group_id(CLASS_NAMES[labels[i]], p)].add(int(split[i]))
assert all(len(v) == 1 for v in seen.values()), "a group ended up in more than one subset"
assert (split >= 0).all()

if QUICK_TEST:                           # tiny subset, only to prove the notebook runs end to end
    keep = []
    for ci in range(len(CLASS_NAMES)):
        for s, n in ((0, 24), (1, 8), (2, 8)):
            keep += list(np.where((labels == ci) & (split == s))[0][:n])
    keep = np.array(sorted(keep))
    files = [files[i] for i in keep]; labels = labels[keep]; split = split[keep]
    print("QUICK_TEST: using", len(files), "images")

NAMES = ["train", "validation", "test"]
print(f"\n{'class':<24}" + "".join(f"{n:>12}" for n in NAMES) + f"{'total':>8}{'groups':>8}")
for ci, c in enumerate(CLASS_NAMES):
    m = labels == ci
    g = len({group_id(c, files[i]) for i in np.where(m)[0]})
    row = [int(((split == s) & m).sum()) for s in range(3)]
    print(f"{c:<24}" + "".join(f"{v:>7} ({v / max(sum(row), 1):>3.0%})" for v in row) + f"{sum(row):>8}{g:>8}")
print(f"{'TOTAL':<24}" + "".join(f"{int((split == s).sum()):>12}" for s in range(3)) + f"{len(files):>8}")

## 4. Load the images exactly like the prediction service
EXIF-straighten, RGB, bilinear resize to 224 x 224, raw 0-255 pixels, no crop (the preprocessing layer is inside the model).

In [ ]:
def load_224(path):
    """Same steps as ai-model/service/app.py: EXIF transpose, RGB, bilinear resize to 224x224, raw 0-255 floats."""
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    with tf.device("/CPU:0"):                  # the service resizes on the CPU; keep the GPU free for training
        return tf.image.resize(np.asarray(img), IMG, method="bilinear").numpy()

def load_cache(path):
    return np.clip(np.rint(load_224(path)), 0, 255).astype(np.uint8)    # uint8 cache keeps RAM small (~150 KB/image)

t0 = time.time()
bad = []
def safe_load(i):
    try:
        return load_cache(files[i])
    except Exception as e:            # an unreadable file must not crash a long run
        bad.append((files[i], repr(e)))
        return None

with ThreadPoolExecutor(8) as ex:
    imgs = list(ex.map(safe_load, range(len(files))))
ok = np.array([a is not None for a in imgs])
print(f"loaded {int(ok.sum())}/{len(files)} images in {time.time() - t0:.0f}s; unreadable: {len(bad)}")
for p, e in bad[:10]:
    print("  UNREADABLE:", p, e)
X = np.stack([a for a in imgs if a is not None])
y = labels[ok]; sp = split[ok]; kept_files = [f for f, k in zip(files, ok) if k]
del imgs

X_tr, y_tr = X[sp == 0], y[sp == 0]
X_va, y_va = X[sp == 1], y[sp == 1]
X_te, y_te = X[sp == 2], y[sp == 2]
test_files = [f for f, s in zip(kept_files, sp) if s == 2]
print("train", X_tr.shape, "| validation", X_va.shape, "| test", X_te.shape)

## 5. Input pipelines (random crop for training only) and class weights

In [ ]:
CROP_PROB, CROP_MIN_AREA = 0.7, 0.6      # random crop: keep 60-100% of the picture, then resize back to 224

def random_crop_resize(img, p=CROP_PROB, min_area=CROP_MIN_AREA):
    """Training-only random crop. With probability p keep a random window (min_area..100% of the image,
    aspect ratio 3:4 .. 4:3) and resize it back to 224x224. It lives in the data pipeline, NOT in the model,
    so the exported model is unchanged: no extra layer, nothing to register, same input contract."""
    def crop():
        area = tf.random.uniform([], min_area, 1.0)
        ratio = tf.exp(tf.random.uniform([], tf.math.log(0.75), tf.math.log(4.0 / 3.0)))
        h = tf.minimum(tf.cast(tf.round(IMG[0] * tf.sqrt(area / ratio)), tf.int32), IMG[0])
        w = tf.minimum(tf.cast(tf.round(IMG[1] * tf.sqrt(area * ratio)), tf.int32), IMG[1])
        top = tf.random.uniform([], 0, IMG[0] - h + 1, dtype=tf.int32)
        left = tf.random.uniform([], 0, IMG[1] - w + 1, dtype=tf.int32)
        return tf.image.resize(tf.image.crop_to_bounding_box(img, top, left, h, w), IMG, method="bilinear")
    return tf.cond(tf.random.uniform([]) < p, crop, lambda: img)

AUTOTUNE = tf.data.AUTOTUNE

def make_train_ds(X_, y_):
    ds = tf.data.Dataset.from_tensor_slices((X_, y_)).shuffle(len(X_), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda a, b: (random_crop_resize(tf.cast(a, tf.float32)), b), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

def make_eval_ds(X_, y_):
    ds = tf.data.Dataset.from_tensor_slices((X_, y_)).map(lambda a, b: (tf.cast(a, tf.float32), b))
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_train_ds(X_tr, y_tr), make_eval_ds(X_va, y_va), make_eval_ds(X_te, y_te)

from sklearn.utils.class_weight import compute_class_weight
w = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_tr)
class_weight = dict(enumerate(w))
print("class_names =", CLASS_NAMES)
print("train images per class:", {CLASS_NAMES[k]: int(v) for k, v in sorted(Counter(y_tr.tolist()).items())})
print("class weights:", {CLASS_NAMES[k]: round(float(v), 3) for k, v in class_weight.items()})

# quick look at what the crop does (first 8 training images, original on top / augmented below)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for j in range(8):
    a = tf.cast(X_tr[j], tf.float32)
    ax[0, j].imshow(X_tr[j]); ax[1, j].imshow(np.clip(random_crop_resize(a, p=1.0).numpy(), 0, 255).astype("uint8"))
    ax[0, j].axis("off"); ax[1, j].axis("off")
ax[0, 0].set_title("original", fontsize=9, loc="left"); ax[1, 0].set_title("random crop", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## 6. Model: MobileNetV2 + the same augmentation layers as before
The preprocessing (`preprocess_input`) and the augmentation layers are INSIDE the model, so the server and phone keep feeding raw 0-255 pixels.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])

base = MobileNetV2(input_shape=IMG + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = layers.Input(shape=IMG + (3,))
x = augment(inputs)
x = preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
model = models.Model(inputs, outputs)
print("input", model.input_shape, "-> output", model.output_shape)

## 7. Train (stage 1: new head only, stage 2: fine-tune)

In [ ]:
history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}

def run_stage(name, epochs, lr, patience):
    ckpt = f"{WORK}/best_{name}.keras"
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    ]
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weight, callbacks=callbacks)
    model.load_weights(ckpt)                      # always continue from / finish with the BEST epoch
    for k in history:
        history[k] += h.history[k]
    best = int(np.argmin(h.history["val_loss"]))
    print(f"[{name}] best epoch {best + 1}/{len(h.history['loss'])}  val_loss={h.history['val_loss'][best]:.4f}  "
          f"val_acc={h.history['val_accuracy'][best]:.4f}")

E1, E2 = (1, 1) if QUICK_TEST else (15, 30)

# --- Stage 1: train only the new top layers
run_stage("stage1_head", epochs=E1, lr=1e-3, patience=5)

# --- Stage 2: fine-tune (first 30 layers and all BatchNorm layers stay frozen)
base.trainable = True
for layer in base.layers[:30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
run_stage("stage2_finetune", epochs=E2, lr=3e-5, patience=5)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["accuracy"], label="train"); ax[0].plot(history["val_accuracy"], label="validation")
ax[0].axvline(E1 - 0.5, color="grey", ls=":"); ax[0].set_title("accuracy (dotted line = fine-tuning starts)"); ax[0].legend()
ax[1].plot(history["loss"], label="train"); ax[1].plot(history["val_loss"], label="validation")
ax[1].axvline(E1 - 0.5, color="grey", ls=":"); ax[1].set_title("loss"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Test-split results: accuracy, precision, recall, F1, confusion matrix
The test split was never used for training, early stopping or checkpoint selection.

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

probs_te = model.predict(test_ds, verbose=0)
y_pred = probs_te.argmax(1)

acc = accuracy_score(y_te, y_pred)
mp, mr, mf1, _ = precision_recall_fscore_support(y_te, y_pred, average="macro", zero_division=0)
print(f"TEST accuracy: {acc:.4f}   macro precision: {mp:.4f}   macro recall: {mr:.4f}   macro F1: {mf1:.4f}   ({len(y_te)} images)\n")
print(classification_report(y_te, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, digits=4, zero_division=0))

corn = y_te != NC
false_reject = int(((y_te != NC) & (y_pred == NC)).sum())
leaked = int(((y_te == NC) & (y_pred != NC)).sum())
print(f"False reject (real corn predicted as Not Corn Leaf): {false_reject}/{int(corn.sum())} = {false_reject / max(int(corn.sum()), 1):.2%}")
print(f"Leaked (Not Corn Leaf predicted as a corn class):    {leaked}/{int((~corn).sum())} = {leaked / max(int((~corn).sum()), 1):.2%}")

cm = confusion_matrix(y_te, y_pred, labels=range(len(CLASS_NAMES)))
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, data, title, fmt in ((ax[0], cm, "Confusion matrix (counts)", "d"), (ax[1], cmn, "Confusion matrix (row %)", ".0%")):
    a.imshow(data, cmap="Blues")
    a.set_xticks(range(len(CLASS_NAMES))); a.set_yticks(range(len(CLASS_NAMES)))
    a.set_xticklabels(CLASS_NAMES, rotation=30, ha="right"); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(title)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            a.text(j, i, format(data[i, j], fmt), ha="center", va="center", color="white" if data[i, j] > data.max() / 2 else "black")
plt.tight_layout()
os.makedirs(EXPORT_DIR, exist_ok=True)
plt.savefig(f"{EXPORT_DIR}/confusion_matrix_test.png", dpi=150)
plt.show()

METRICS = dict(test_images=int(len(y_te)), accuracy=round(float(acc), 4), macro_precision=round(float(mp), 4),
               macro_recall=round(float(mr), 4), macro_f1=round(float(mf1), 4),
               false_reject_rate=round(false_reject / max(int(corn.sum()), 1), 4))
print(METRICS)

## 9. Export: `.keras`, `.tflite`, `labels.json` (same names and format as before)

In [ ]:
import tensorflow as tf, keras

def to_label(name):                       # training folder name -> label used by the server / phone
    n = name.lower()
    if "not corn" in n: return "not_corn"
    if "rust" in n: return "southern_rust"
    if "gray" in n: return "gray_leaf_spot"
    if "blight" in n: return "northern_leaf_blight"
    if "healthy" in n: return "healthy"
    raise ValueError(name)

LABELS = [to_label(c) for c in CLASS_NAMES]      # follows the model OUTPUT order
assert sorted(LABELS) == sorted(["southern_rust", "gray_leaf_spot", "healthy", "northern_leaf_blight", "not_corn"])
assert len(MODEL_VERSION) <= 50

os.makedirs(EXPORT_DIR, exist_ok=True)
KERAS_OUT = f"{EXPORT_DIR}/leafscan_mobilenetv2.keras"
TFLITE_OUT = f"{EXPORT_DIR}/leafscan_mobilenetv2.tflite"

model.save(KERAS_OUT)
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
with open(TFLITE_OUT, "wb") as f:
    f.write(conv.convert())

meta = {
    "model_name": "MobileNetV2 (transfer learning, fine-tuned) — 5 classes with not_corn",
    "model_version": MODEL_VERSION,
    "class_labels": LABELS,
    "training_folder_names": CLASS_NAMES,
    "rejection_class": "not_corn",
    "input_size": [224, 224],
    "color_mode": "RGB",
    "input_range": "raw_0_255",
    "note": "Preprocessing is INSIDE the model. Do NOT divide by 255 or normalize. Resize bilinear to 224x224, no crop.",
    "confidence_levels": {"low": [0, 39], "moderate": [40, 69], "high": [70, 100]},
    "test_results": METRICS,
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
}
with open(f"{EXPORT_DIR}/labels.json", "w") as f:
    json.dump(meta, f, indent=2)

# Extra files for the record (NOT needed by the server or the phone)
with open(f"{EXPORT_DIR}/split_manifest.csv", "w", newline="", encoding="utf-8") as f:
    wr = csv.writer(f); wr.writerow(["class", "filename", "group", "split"])
    for p, l, s in zip(kept_files, y, sp):
        wr.writerow([CLASS_NAMES[l], os.path.basename(p), group_id(CLASS_NAMES[l], p), NAMES[s]])
report = {"seed": SEED, "split": SPLIT, "class_names": CLASS_NAMES,
          "images_per_class": {c: int((y == i).sum()) for i, c in enumerate(CLASS_NAMES)},
          "train_class_weights": {CLASS_NAMES[k]: float(v) for k, v in class_weight.items()},
          "test_metrics": METRICS, "classification_report": classification_report(
              y_te, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, output_dict=True, zero_division=0),
          "confusion_matrix": cm.tolist(), "quick_test": QUICK_TEST}
with open(f"{EXPORT_DIR}/training_report.json", "w") as f:
    json.dump(report, f, indent=2)

# --- Keras vs TFLite must agree
def tflite_predict(path, arr):
    itp = tf.lite.Interpreter(model_path=path); itp.allocate_tensors()
    i_, o_ = itp.get_input_details()[0], itp.get_output_details()[0]
    out = []
    for a in arr:
        itp.set_tensor(i_["index"], a[None].astype(np.float32)); itp.invoke()
        out.append(itp.get_tensor(o_["index"])[0].copy())
    return np.array(out)

rs = np.random.default_rng(SEED)
rand = rs.uniform(0, 255, size=(5, 224, 224, 3)).astype(np.float32)
m5 = int((model.predict(rand, verbose=0).argmax(1) == tflite_predict(TFLITE_OUT, rand).argmax(1)).sum())
Xf = X_te.astype(np.float32)
tfl_te = tflite_predict(TFLITE_OUT, Xf)
print(f"Keras vs TFLite on 5 random inputs: {m5}/5 same")
print(f"Keras vs TFLite on the {len(Xf)} test images: {np.mean(tfl_te.argmax(1) == y_pred):.2%} same | "
      f"TFLite test accuracy: {accuracy_score(y_te, tfl_te.argmax(1)):.4f}")
print("\nFiles in", EXPORT_DIR)
for fn in sorted(os.listdir(EXPORT_DIR)):
    print(f"  {os.path.getsize(os.path.join(EXPORT_DIR, fn)) / 1e6:8.2f} MB  {fn}")

## 10. Field-test photos (real corn leaves, never trained on): how many are rejected as `not_corn`?
Put ONLY real corn-leaf photos in `FIELD_TEST_DIR` (cell 1). Optional: put them in sub-folders named like the classes
(`Healthy`, `Gray Leaf Spot`, ...) and the cell also reports accuracy per class.
The server also has a separate colour check that can reject a photo before the model sees it; that check is not simulated here.

In [ ]:
FIELD_EXT = (".jpg", ".jpeg", ".png", ".webp")

if not os.path.isdir(FIELD_TEST_DIR):
    print(f"Field-test folder not found: {FIELD_TEST_DIR}\nUpload the field-test folder to Drive and fix FIELD_TEST_DIR in cell 1.")
else:
    paths = sorted(os.path.join(dp, fn) for dp, _, fns in os.walk(FIELD_TEST_DIR) for fn in fns if fn.lower().endswith(FIELD_EXT))
    other = sum(1 for dp, _, fns in os.walk(FIELD_TEST_DIR) for fn in fns if not fn.lower().endswith(FIELD_EXT))
    print(f"{len(paths)} field photos found" + (f" ({other} non-image files ignored; HEIC is not supported - convert to JPG)" if other else ""))
    assert paths, "no images in the field-test folder"

    # Safety: none of these photos may also be in the training data
    md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
    train_md5 = {md5(p) for p in files}
    overlap = [p for p in paths if md5(p) in train_md5]
    print("field photos that are byte-identical to a TRAINING image:", len(overlap), overlap[:5])

    key = lambda s: s.lower().replace("_", " ").replace("-", " ").strip()
    by_key = {key(c): i for i, c in enumerate(CLASS_NAMES)}
    true_idx = [by_key.get(key(os.path.basename(os.path.dirname(p))), None) for p in paths]

    Xf = np.stack([load_224(p) for p in paths])                  # exact service-style preprocessing, no rounding
    p_keras = model.predict(Xf, batch_size=32, verbose=0)
    p_tfl = tflite_predict(TFLITE_OUT, Xf)                       # what the phone's on-device model would say

    def summarise(name, probs):
        pred = probs.argmax(1); conf = probs.max(1) * 100
        rej = np.where(pred == NC)[0]
        print(f"\n=== {name} ===")
        print("predicted class counts:", {CLASS_NAMES[k]: int(v) for k, v in sorted(Counter(pred.tolist()).items())})
        print(f"confidence: high (>=70) {int((conf >= 70).sum())} | moderate (40-69) {int(((conf >= 40) & (conf < 70)).sum())} | low (<40) {int((conf < 40).sum())}")
        print(f">>> REAL CORN PHOTOS REJECTED AS not_corn: {len(rej)} / {len(paths)} = {len(rej) / len(paths):.1%}")
        for i in rej:
            print(f"      rejected: {os.path.relpath(paths[i], FIELD_TEST_DIR)}  (not_corn {probs[i, NC] * 100:.0f}%)")
        known = [i for i, t in enumerate(true_idx) if t is not None]
        if known:
            ok_ = sum(pred[i] == true_idx[i] for i in known)
            print(f"accuracy on the {len(known)} photos with a class sub-folder: {ok_}/{len(known)} = {ok_ / len(known):.1%}")
            for ci, c in enumerate(CLASS_NAMES):
                sel = [i for i in known if true_idx[i] == ci]
                if sel:
                    print(f"      {c:<22} {sum(pred[i] == ci for i in sel)}/{len(sel)} correct")

    summarise("Keras model (server)", p_keras)
    summarise("TFLite model (phone, offline)", p_tfl)